# PB-06 · Prototipo interactivo con `ipywidgets`

Este notebook implementa el prototipo interactivo solicitado en la historia de usuario **PB-06** para explorar el dataset **Online Shoppers Purchasing Intention**.

## Qué incluye
- Carga y vista rápida del dataset
- Identificación de variables numéricas y categóricas
- Controles interactivos con:
  - **Dropdown** para elegir variable
  - **Slider** para ajustar bins
  - **Checkbox** para activar/desactivar boxplot adicional
  - **RadioButtons** para elegir el tipo de gráfico
- Filtro opcional por la variable objetivo `Revenue`
- Resumen estadístico automático según la variable seleccionada

> **Nota:** coloca este notebook en la misma carpeta que `online_shoppers_intention.csv`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ipywidgets import interact, Dropdown, IntSlider, Checkbox, RadioButtons
import ipywidgets as widgets
from IPython.display import display, Markdown

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


In [ ]:
# Cargar datos
df = pd.read_csv("online_shoppers_intention.csv")

print(f"Dataset cargado correctamente: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()


In [ ]:
# Tipos de variables
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]

print("Variables numéricas:")
print(num_cols)
print("\nVariables categóricas:")
print(cat_cols)


## Exploración base del dataset

Esta sección ayuda a documentar el notebook, tal como pide PB-06, y deja evidencia de comprensión del conjunto de datos.


In [ ]:
# Resumen general
display(df.describe(include="all").T.fillna(""))


## Función interactiva

La función:
- filtra por `Revenue` si el usuario lo desea,
- genera distintos tipos de gráficos según el tipo de variable,
- muestra un boxplot adicional si se activa el checkbox,
- imprime un pequeño resumen estadístico o de frecuencias.


In [ ]:
def explorar(variable, bins, mostrar_boxplot, tipo_grafico, filtro_revenue):
    data = df.copy()

    # Filtro opcional por target
    if filtro_revenue != "Todos":
        valor = True if filtro_revenue == "True" else False
        data = data[data["Revenue"] == valor]

    if data.empty:
        print("No hay datos para el filtro seleccionado.")
        return

    es_numerica = variable in num_cols
    es_categorica = variable in cat_cols

    plt.figure(figsize=(10, 5))

    if tipo_grafico == "Histograma":
        if es_numerica:
            sns.histplot(data[variable].dropna(), bins=bins, kde=True)
            plt.title(f"Histograma de {variable}")
            plt.xlabel(variable)
            plt.ylabel("Frecuencia")
        else:
            print(f"La variable '{variable}' no es numérica. Usa 'Conteo' para variables categóricas.")
            plt.close()

    elif tipo_grafico == "Boxplot":
        if es_numerica:
            sns.boxplot(x=data[variable])
            plt.title(f"Boxplot de {variable}")
            plt.xlabel(variable)
        else:
            print(f"La variable '{variable}' no es numérica. El boxplot aplica a variables numéricas.")
            plt.close()

    elif tipo_grafico == "Conteo":
        if es_categorica:
            order = data[variable].astype(str).value_counts().index
            sns.countplot(x=data[variable].astype(str), order=order)
            plt.xticks(rotation=45, ha="right")
            plt.title(f"Conteo de {variable}")
            plt.xlabel(variable)
            plt.ylabel("Frecuencia")
        else:
            print(f"La variable '{variable}' es numérica. Usa histograma o boxplot.")
            plt.close()
    else:
        print("Tipo de gráfico no reconocido.")
        plt.close()

    if plt.get_fignums():
        plt.tight_layout()
        plt.show()

    # Boxplot adicional opcional
    if mostrar_boxplot and es_numerica and tipo_grafico != "Boxplot":
        plt.figure(figsize=(8, 2.8))
        sns.boxplot(x=data[variable])
        plt.title(f"Boxplot adicional de {variable}")
        plt.xlabel(variable)
        plt.tight_layout()
        plt.show()

    # Resumen textual
    print("\nResumen de la variable:")
    if es_numerica:
        display(data[variable].describe().to_frame(name=variable))
    elif es_categorica:
        display(data[variable].astype(str).value_counts().to_frame(name="frecuencia"))


## Interfaz interactiva

Este bloque constituye el **prototipo PB-06**: un stakeholder no técnico puede explorar los datos sin escribir código.


In [ ]:
interact(
    explorar,
    variable=Dropdown(
        options=df.columns.tolist(),
        value=df.columns.tolist()[0],
        description="Variable:"
    ),
    bins=IntSlider(
        min=5, max=100, step=5, value=30,
        description="Bins:"
    ),
    mostrar_boxplot=Checkbox(
        value=False,
        description="Boxplot extra"
    ),
    tipo_grafico=RadioButtons(
        options=["Histograma", "Boxplot", "Conteo"],
        value="Histograma",
        description="Gráfico:"
    ),
    filtro_revenue=RadioButtons(
        options=["Todos", "True", "False"],
        value="Todos",
        description="Revenue:"
    )
);


## Sugerencias de uso en la exposición

Puedes demostrar el prototipo así:
1. Seleccionar una variable numérica, por ejemplo `PageValues`.
2. Cambiar los `bins` del histograma.
3. Activar el **boxplot extra** para ver dispersión y outliers.
4. Cambiar el filtro `Revenue` para comparar sesiones que compraron vs no compraron.
5. Elegir una variable categórica como `Month` o `VisitorType` y mostrar el gráfico de conteo.

## Conclusión
Este notebook cumple la historia **PB-06** porque ofrece una exploración visual e interactiva del dataset mediante `ipywidgets`, accesible para usuarios no técnicos.
